In [32]:
import pandas as pd
import json
import os
import numpy as np


In [ ]:
def build_ticker_date_mapping(parquet_file: str, output_json: str):
    """
    Passes through the entire dataset ONE time and builds a master dictionary mapping
    every unique ticker to the exact days it was mentioned.
    Format: {"AAPL": ["2023-01-01", "2023-01-05"], "TSLA": [...]}
    """
    df = pd.read_parquet(parquet_file)
    df['datetime'] = pd.to_datetime(df['datetime']).dt.strftime('%Y-%m-%d')

    # splits tickers mentioned on the same day into different rows
    exploded_df = df.explode('tickers')
    exploded_df = exploded_df.dropna(subset=['tickers'])

    # finds unique() dates and converts them to a list, and turns the whole thing into dict
    ticker_date_dict = exploded_df.groupby('tickers')['datetime'].unique().apply(list).to_dict()

    print(f"saving to {output_json}...")
    with open(output_json, 'w') as f:
        json.dump(ticker_date_dict, f, indent=4)
        
    print(f"total unique stocks mentioned: {len(ticker_date_dict):,}")
    
    # sort the dictionary by the length of the date lists
    top_stocks = sorted(ticker_date_dict.items(), key=lambda x: len(x[1]), reverse=True)[:10]
    print("\nTop 10 Most Discussed Stocks (By Unique Days):")
    for stock, dates in top_stocks:
        print(f"   ${stock}: {len(dates):,} unique days of discussion")

if __name__ == "__main__":
    PARQUET_FILE = "wsb_submissions_ml_ready.parquet"
    OUTPUT_JSON = "ticker_date_mapping.json"
    
    if os.path.exists(PARQUET_FILE):
        build_ticker_date_mapping(PARQUET_FILE, OUTPUT_JSON)
    else:
        print(f"Could not find {PARQUET_FILE}")

saving to ticker_date_mapping.json...
total unique stocks mentioned: 4,932

Top 10 Most Discussed Stocks (By Unique Days):
   $TSLA: 1,722 unique days of discussion
   $AMD: 1,540 unique days of discussion
   $EPS: 1,530 unique days of discussion
   $AAPL: 1,308 unique days of discussion
   $NVDA: 1,285 unique days of discussion
   $AMZN: 1,054 unique days of discussion
   $MSFT: 857 unique days of discussion
   $RSI: 762 unique days of discussion
   $QQQ: 752 unique days of discussion
   $GME: 709 unique days of discussion


In [11]:
## finding reason for missing data

import pandas as pd

def investigate_time_travel(raw_parquet_file: str):
    df = pd.read_parquet(raw_parquet_file)
    
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['year'] = df['datetime'].dt.year

    print("\nraw data - posts/yr:")
    print(df['year'].value_counts().sort_index())
    
    missing_ratio = df['upvote_ratio'].isna().sum()
    print(f"\nposts with no upvote_ratio data: {missing_ratio:,}")
    
    old_posts = df[df['year'] < 2020].copy()
    old_posts['flair'] = old_posts['flair'].fillna('')
    has_dd_flair = old_posts[old_posts['flair'].isin(['DD', 'Discussion', 'Fundamentals', 'Chart'])]
    
    print(f"\nout of {len(old_posts):,} posts before 2020")
    print(f"only {len(has_dd_flair):,} used modern flairs")

if __name__ == "__main__":
    RAW_FILE = "wsb_submissions_clean.parquet" 
    import os
    if os.path.exists(RAW_FILE):
        investigate_time_travel(RAW_FILE)
    else:
        print("raw file not found")


raw data - posts/yr:
year
2012        610
2013        849
2014       2140
2015      11944
2016      37667
2017      49590
2018      89006
2019      70597
2020     303857
2021    1407544
2022     244439
2023     130882
2024     169305
2025     125640
Name: count, dtype: int64

posts with no upvote_ratio data: 0

out of 262,403 posts before 2020
only 29,116 used modern flairs


In [31]:
## NEW FILTER FOR DATA TO PREVENT FLAIR ERROR
import pandas as pd
from datetime import datetime
import re
import nltk
import os
from nltk.corpus import words
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
from polygon import RESTClient
from dotenv import load_dotenv


# Enable progress bars for pandas
tqdm.pandas()

load_dotenv()
polygon_key = os.getenv("POLYGON_API_KEY")
client = RESTClient(polygon_key)

try:
    nltk.data.find('corpora/words')
except LookupError:
    nltk.download('words')
    
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
    nltk.download('omw-1.4')

# Memory cache to prevent duplicate API network requests
ticker_cache = {}

ENGLISH_DICTIONARY = set(word.upper() for word in words.words())

def check_if_ticker_valid(word):
    try:
        # sends request to Massive API to determine whether the current word is a valid ticker
        details = client.get_ticker_details(word)
        return True
    except Exception as e:
        return False
    return False

def filter_wsb_posts(df: pd.DataFrame) -> pd.DataFrame:
    print(f"Initial rows before filtering: {len(df):,}")
    
    df['text'] = df['text'].fillna('')
    if 'flair' in df.columns:
        df['flair'] = df['flair'].fillna('')
    else:
        df['flair'] = ''
    
    df = df[~df['text'].str.isspace()]
    junk_texts = ['[deleted]', '[removed]', 'deleted', 'removed', '']
    df = df[~df['text'].str.strip().str.lower().isin(junk_texts)]
    
    df = df[(df['score'] >= 5) ]
    df = df[df['text'].str.len() >= 75]
    
    # TIME FILTER
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    df['year'] = df['datetime'].dt.year
    
    is_old_school = df['year'] < 2020

    valid_flairs = ['DD', 'Discussion', 'Fundamentals', 'Chart']
    has_valid_flair = df['flair'].isin(valid_flairs)
    has_good_ratio = df['upvote_ratio'] > 0.5
    
    df = df[is_old_school | (has_valid_flair & has_good_ratio)]
    
    df = df.drop(columns=['year']).reset_index(drop=True)
    print(f"filter complete - Remaining posts: {len(df):,}")
    
    return df

def smart_extract_tickers(text):
    blacklist = {
        "YOLO", "TOS", "CEO", "CFO", "CTO", "DD", "BTFD", "WSB", "OK", "RH",
        "KYS", "FD", "TYS", "US", "USA", "IT", "ATH", "RIP", "BMW", "GDP",
        "OTM", "ATM", "ITM", "IMO", "LOL", "DOJ", "BE", "PR", "PC", "ICE",
        "ISIS", "PRAY", "PT", "FBI", "SEC", "GOD", "NOT", "POS", "COD",
        "AYYMD", "FOMO", "TLDR", "EDIT", "STILL", "LGMA", "WTF", "RAW", "PM",
        "LMAO", "LMFAO", "ROFL", "EZ", "RED", "BEZOS", "TICK", "IS", "DOW",
        "AM", "LPT", "GOAT", "FL", "CA", "IL", "PDFUA", "MACD", "HQ",
        "OP", "DJIA", "PS", "AH", "TL", "DR", "JAN", "FEB", "JUL", "AUG",
        "SEP", "SEPT", "OCT", "NOV", "DEC", "FDA", "IV", "ER", "IPO", "RISE",
        "IPA", "URL", "MILF", "BUT", "SSN", "FIFA", "USD", "CPU", "AT",
        "GG", "ELON", "WE", "OMG", "GUH", "HODL", "HOLD", "ROPE"
    }
    
    raw_matches = re.findall(r'\$?[A-Z]{2,5}\b', str(text)) # updated to 2-5 letter tickers to prevent bad data
    potential_tickers = set()
    
    for match in raw_matches:
        has_dollar_sign = match.startswith('$')
        clean_ticker = match.replace('$', '')
        
        if clean_ticker in blacklist: continue
        
        # Cash-tags
        if has_dollar_sign:
            potential_tickers.add(clean_ticker)
            continue
            
        # Layer 2 Defense (Length & English Dictionary)
        if len(clean_ticker) <= 2: continue
        if clean_ticker in ENGLISH_DICTIONARY: continue
            
        potential_tickers.add(clean_ticker)
        
    # API Validation with Memory Caching
    valid_tickers = []
    for ticker in potential_tickers:
        if ticker in ticker_cache:
            if ticker_cache[ticker]:
                valid_tickers.append(ticker)
        else:
            # If not in cache, ask the API
            if check_if_ticker_valid(ticker):
                ticker_cache[ticker] = True
                valid_tickers.append(ticker)
            else:
                ticker_cache[ticker] = False
                
    return valid_tickers

def clean_and_lemmatize(text, lemmatizer):
    """
    Cleans the text but strictly PRESERVES tickers and the $ symbol.
    """
    text = str(text)
    # remove web links 
    text = re.sub(r'http\S+', '', text)        
    
    # remove punctuation, keep letters, spaces, and $ sign
    text = re.sub(r'[^a-zA-Z\s]', '', text)    
    
    # LEMMATIZATION
    # Convert to lowercase and split into words
    words = text.lower().split()
    
    # Pass each word through the lemmatizer
    return ' '.join([lemmatizer.lemmatize(w) for w in words])

def updated_process_dataset(input_file: str, output_file: str):
    print(f"loading dataset: {input_file}")
    df = pd.read_parquet(input_file)
    
    df = filter_wsb_posts(df)
    if len(df) == 0:
        print("ERROR: no rows left")
        return
    df['tickers'] = df['text'].progress_apply(smart_extract_tickers)
    
    # Drop rows that have ZERO valid tickers
    df = df[df['tickers'].str.len() > 0].copy()
    print(f"Posts with valid tickers remaining: {len(df):,}")
    
    print("\n Lemmatizing Text...")
    lemmatizer = WordNetLemmatizer()
    df['clean_text'] = df['text'].progress_apply(lambda x: clean_and_lemmatize(x, lemmatizer))
    
    print(f"\n Saving final ML-Ready dataset")
    columns_to_keep = ['datetime', 'score', 'tickers', 'clean_text', 'flair', 'upvote_ratio']
    
    # Prevent KeyErrors by only keeping columns that exist
    columns_to_keep = [col for col in columns_to_keep if col in df.columns]
        
    final_df = df[columns_to_keep]
    final_df.to_parquet(output_file, engine='pyarrow', index=False)

if __name__ == "__main__":
    INPUT_DATA = "wsb_submissions_clean.parquet" 
    
    OUTPUT_DATA = "wsb_submissons_ml_ready.parquet"
    
    if os.path.exists(INPUT_DATA):
        updated_process_dataset(INPUT_DATA, OUTPUT_DATA)
    else:
        print(f"Error: Cannot find {INPUT_DATA}.")

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/thibautchen/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/thibautchen/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


loading dataset: wsb_submissions_clean.parquet
Initial rows before filtering: 2,644,070
filter complete - Remaining posts: 128,363


100%|██████████| 128363/128363 [36:17<00:00, 58.95it/s]  


Posts with valid tickers remaining: 58,466

 Lemmatizing Text...


100%|██████████| 58466/58466 [00:23<00:00, 2489.18it/s]



 Saving final ML-Ready dataset


In [16]:
'''
Testing smart_extract_tickers and check_if_ticker_valid
'''
text1 = "I am buying $AAPL and TSLA plus $F and C tomorrow, OMG AAPL is so ICE. YOLO! KKK" #end -> test check_if_ticker_valid function
print(text1)
print()
print(smart_extract_tickers(text1))
ticker_valid_test_list = []
for word in smart_extract_tickers(text1):
    ticker_valid_test_list.append(check_if_ticker_valid(word))
    
ticker_valid_test_list

I am buying $AAPL and TSLA plus $F and C tomorrow, OMG AAPL is so ICE. YOLO! KKK

['TSLA', 'AAPL']


[True, True]

In [19]:
'''
Testing cleaning and lemmatization
'''
lemmatizer = WordNetLemmatizer()
    
raw_text = "I am BUYING 100 shares of $NVDA!!! Check this link: https://google.com 🚀"
cleaned = clean_and_lemmatize(raw_text, lemmatizer)

expected = "i am buying share of nvda check this link"
    
assert cleaned == expected, f"Lemmatization failed.\nExpected: {expected}\nGot: {cleaned}"
print("clean_and_lemmatize test passes\n")

clean_and_lemmatize test passes



In [23]:
'''
Testing new filter functon
'''
# Create mock Reddit data covering all edge cases
mock_data = pd.DataFrame({
        'text': [
            "This is a fantastic Due Diligence post about Apple that is very long and has lots of good information.", # 0: valid modern
            "[removed]", # 1: invalid: removed
            "   ", # 2: invalid: blank spaces
            "Short post", # 3: invalid: under character count
            "This is a great long post but everyone hated it so it got downvoted to oblivion.", # 4: invalid: score < 5
            "This post is long enough and has a good score, but the upvote ratio is terrible due to controversy.", # 5: invalid: Ratio < 0.5
            "Valid post length and score but missing a flair because the user forgot to tag it.", # 6: invalid modern: No Flair
            "Valid post length and score but its a stupid ahh meme so its a shitpost lololol.", # 7: invalid modern: shitpost
            "This is a really long, high quality post from 2018. Back then, WallStreetBets didn't really use the modern flair system, so the flair is blank.", # 8: valid old-school: No flair
            "This is another really long, high quality post from 2017. It has a weird old flair that isn't DD, but we keep it because it is from 2017.", # 9: valid old-school: Weird flair
            "Short old post." # 10: invalid old-school: Under 75 chars
        ],
        'score': [100, 10, 10, 10, 2, 100, 100, 100, 100, 100, 100],
        'upvote_ratio': [0.95, 0.9, 0.9, 0.9, 0.9, 0.4, 0.9, 0.9, 0.9, 0.9, 0.9],
        'flair': ['DD', 'DD', 'DD', 'DD', 'DD', 'DD', '', 'shitpost', '', 'Old Meme', ''],
        'datetime': [
            '2021-05-01 12:00:00', 
            '2021-05-01 12:00:00', 
            '2021-05-01 12:00:00', 
            '2021-05-01 12:00:00',
            '2021-05-01 12:00:00', 
            '2021-05-01 12:00:00',
            '2021-05-01 12:00:00', 
            '2021-05-01 12:00:00',
            '2018-05-01 12:00:00', # 8: Old School (< 2020)
            '2017-05-01 12:00:00', # 9: Old School (< 2020)
            '2018-05-01 12:00:00'  # 10: Old School (< 2020)
        ]
    })
    
filtered_df = filter_wsb_posts(mock_data)
    
# We expect exactly 3 posts to survive: Index 0, Index 8, and Index 9.
assert len(filtered_df) == 3, f"Filter failed! Expected 3 posts to survive, but got {len(filtered_df)}."
    
# Verify the exact posts that survived
surviving_texts = filtered_df['text'].tolist()
assert "This is a fantastic Due Diligence post" in surviving_texts[0], "Failed to keep valid modern post."
assert "Back then, WallStreetBets didn't really use the modern flair" in surviving_texts[1], "Failed to keep old-school no-flair post."
assert "It has a weird old flair that isn't DD" in surviving_texts[2], "Failed to keep old-school weird-flair post."
    
print("filter_wsb_posts passed all time-aware tests!\n")

Initial rows before filtering: 11
filter complete - Remaining posts: 3
filter_wsb_posts passed all time-aware tests!



In [35]:
import pandas as pd

df = pd.read_parquet('ml_training_data.parquet')
df.tail(50).to_csv('sample_data.csv', index=False)